# Multi-Statement Financial Extractor
## Extract Income Statement, Balance Sheet, and Cash Flow from SEC Filings

**Features:**
- ✅ Extracts all 3 financial statements
- ✅ AI-powered concept mapping
- ✅ DuckDB logging and tracking
- ✅ Cross-statement validation
- ✅ Export to CSV and Excel

---

## Table of Contents
1. [Setup & Configuration](#section1)
2. [Test Imports & Verify Setup](#section2)
3. [Download SEC Filing](#section3)
4. [Extract Income Statement](#section4)
5. [Extract Balance Sheet](#section5)
6. [Extract Cash Flow Statement](#section6)
7. [Combined Analysis & Metrics](#section7)
8. [Export All Statements](#section8)
9. [Batch Processing (Multiple Tickers)](#section9)

---
## 1. Setup & Configuration
**Purpose:** Import libraries and set configuration

**Troubleshooting:** If this fails, check:
- .env file exists with SEC_ID
- All packages installed
- File structure correct

In [1]:
# Core imports
import os
import sys
from pathlib import Path
from datetime import datetime, date
from dotenv import load_dotenv
import pandas as pd
from edgar import Company, set_identity
from typing import Optional, Literal
import asyncio

# Load environment variables
load_dotenv()

# Set SEC identity
sec_identity = os.getenv("SEC_ID")
if not sec_identity:
    raise ValueError("SEC_ID not found in .env file")
set_identity(sec_identity)

print("✓ Core imports loaded")
print(f"✓ SEC identity configured: {sec_identity.split()[0]}")

✓ Core imports loaded
✓ SEC identity configured: pedroemail@duck.com


/home/pedro/projects/fin_import2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ============================================================================
# CONFIGURATION - CHANGE THESE VALUES
# ============================================================================

TICKER = "AAPL"              # Company ticker symbol
FILING_TYPE = "10-K"         # "10-K" (annual) or "10-Q" (quarterly)
YEAR = 2024                  # Fiscal year (None = most recent)
QUARTER = None               # 1-4 for 10-Q, None for 10-K
USE_AI_FALLBACK = True       # Enable AI for unmapped concepts

# Output directory
OUTPUT_DIR = "./financial_statements"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# DuckDB database path
DB_PATH = "data/xbrl_mappings_multi.duckdb"

# Display configuration
print("="*80)
print("MULTI-STATEMENT FINANCIAL EXTRACTOR")
print("="*80)
print(f"\nConfiguration:")
print(f"  Ticker:         {TICKER}")
print(f"  Filing Type:    {FILING_TYPE}")
print(f"  Year:           {YEAR if YEAR else 'Most Recent'}")
if FILING_TYPE == "10-Q":
    print(f"  Quarter:        Q{QUARTER if QUARTER else 'Most Recent'}")
print(f"  AI Fallback:    {'Enabled' if USE_AI_FALLBACK else 'Disabled'}")
print(f"  Output Dir:     {OUTPUT_DIR}")
print(f"  Database:       {DB_PATH}")
print()

---
## 2. Test Imports & Verify Setup
**Purpose:** Verify all extractors and mappings are available

**Troubleshooting:** If this fails:
- Check `extractors/` folder exists
- Check `xbrl_mappings/` folder exists with `__init__.py`
- Verify all 3 extractor files are present

In [ ]:
# Test 1: Import extractors
print("Testing extractor imports...")
print("-" * 80)

try:
    from extractors.income_statement_extractor import (
        get_filing,
        extract_income_statement
    )
    print("✓ Income statement extractor loaded")
except ImportError as e:
    print(f"✗ Income statement extractor failed: {e}")
    raise

try:
    from extractors.balance_sheet_extractor import extract_balance_sheet
    print("✓ Balance sheet extractor loaded")
except ImportError as e:
    print(f"✗ Balance sheet extractor failed: {e}")
    raise

try:
    from extractors.cash_flow_extractor import extract_cash_flow
    print("✓ Cash flow extractor loaded")
except ImportError as e:
    print(f"✗ Cash flow extractor failed: {e}")
    raise

In [ ]:
# Test 2: Import mappings
print("\nTesting mapping imports...")
print("-" * 80)

try:
    from xbrl_mappings import (
        INCOME_STATEMENT_MAPPING,
        BALANCE_SHEET_MAPPING,
        CASH_FLOW_MAPPING
    )
    
    income_concepts = sum(len(v) for v in INCOME_STATEMENT_MAPPING.values())
    balance_concepts = sum(len(v) for v in BALANCE_SHEET_MAPPING.values())
    cashflow_concepts = sum(len(v) for v in CASH_FLOW_MAPPING.values())
    
    print(f"✓ Income statement:  {len(INCOME_STATEMENT_MAPPING)} fields, {income_concepts} concepts")
    print(f"✓ Balance sheet:     {len(BALANCE_SHEET_MAPPING)} fields, {balance_concepts} concepts")
    print(f"✓ Cash flow:         {len(CASH_FLOW_MAPPING)} fields, {cashflow_concepts} concepts")
    print(f"\n✓ Total: {len(INCOME_STATEMENT_MAPPING) + len(BALANCE_SHEET_MAPPING) + len(CASH_FLOW_MAPPING)} fields, {income_concepts + balance_concepts + cashflow_concepts} concepts")
    
except ImportError as e:
    print(f"✗ Mapping import failed: {e}")
    print("\nMake sure xbrl_mappings/ folder has:")
    print("  - __init__.py")
    print("  - income_statement_xbrl_mapping.py")
    print("  - balance_sheet_xbrl_mapping.py")
    print("  - cash_flow_xbrl_mapping.py")
    raise

In [ ]:
# Test 3: Test database connection (optional)
print("\nTesting database connection...")
print("-" * 80)

try:
    from xbrl_mapping_manager_multi_statement import XBRLMappingManager
    
    # Initialize (creates DB if doesn't exist)
    mapper = XBRLMappingManager(DB_PATH)
    
    # Test query
    health = mapper.conn.execute("""
        SELECT COUNT(*) as count FROM core_concept_mappings
    """).fetchone()
    
    print(f"✓ Database connected: {DB_PATH}")
    print(f"✓ Core mappings in DB: {health[0]}")
    
    mapper.close()
    
except Exception as e:
    print(f"⚠ Database connection failed: {e}")
    print("  This is optional - extraction will still work")

print("\n" + "="*80)
print("✓ ALL SYSTEMS READY")
print("="*80)

---
## 3. Download SEC Filing
**Purpose:** Download the filing from EDGAR

**Troubleshooting:** If this fails:
- Check ticker symbol is valid
- Check year has filings
- Check internet connection
- Check SEC_ID in .env

In [ ]:
print("="*80)
print("DOWNLOADING SEC FILING")
print("="*80)

try:
    filing = get_filing(TICKER, FILING_TYPE, YEAR, QUARTER)
    
    print(f"\n✓ Filing downloaded successfully!")
    print(f"\nFiling Details:")
    print(f"  Company:        {filing.company if hasattr(filing, 'company') else 'N/A'}")
    print(f"  Form:           {filing.form if hasattr(filing, 'form') else FILING_TYPE}")
    print(f"  Filing Date:    {filing.filing_date if hasattr(filing, 'filing_date') else 'N/A'}")
    print(f"  Period:         {filing.period_of_report if hasattr(filing, 'period_of_report') else 'N/A'}")
    
except Exception as e:
    print(f"\n✗ Failed to download filing: {e}")
    raise

---
## 4. Extract Income Statement
**Purpose:** Extract all income statement line items

**Expected:** 25-28 out of 30 fields for most companies

**Troubleshooting:** If extraction fails:
- Check filing has XBRL data
- Check AI mapper is working (if enabled)
- Try with USE_AI_FALLBACK = False

In [ ]:
# Extract income statement
income_df = await extract_income_statement(
    filing, 
    TICKER, 
    FILING_TYPE,
    YEAR,
    QUARTER,
    use_ai_fallback=USE_AI_FALLBACK
)

# Calculate coverage
income_found = len(income_df[income_df['Status'] == '✓'])
income_total = len(income_df)
income_coverage = (income_found / income_total) * 100

print(f"\n{'='*80}")
print(f"INCOME STATEMENT RESULTS")
print(f"{'='*80}")
print(f"\nData Quality: {income_found}/{income_total} fields ({income_coverage:.1f}%)")

# Show key metrics
print(f"\nKey Metrics:")
key_fields = ['revenue', 'gross_profit', 'operating_income', 'net_income', 'diluted_eps']
for field in key_fields:
    row = income_df[income_df['Field'] == field]
    if not row.empty and pd.notna(row.iloc[0]['Value']):
        value = row.iloc[0]['Value']
        if field == 'diluted_eps':
            print(f"  {field:20s}: ${value:,.2f}")
        else:
            print(f"  {field:20s}: ${value:,.0f}")

In [ ]:
# Display income statement by section
print("\n" + "="*80)
print("INCOME STATEMENT BREAKDOWN")
print("="*80)

sections = {
    "Revenue & Costs": ['revenue', 'cost_of_revenue', 'gross_profit'],
    "Operating Expenses": ['research_development', 'selling_general_admin', 'depreciation_amortization', 
                          'total_operating_expenses', 'operating_income'],
    "Non-Operating Items": ['interest_income', 'interest_expense', 'other_nonoperating_income'],
    "Net Income": ['pretax_income', 'income_tax_expense', 'net_income'],
    "Per Share": ['basic_eps', 'diluted_eps', 'basic_shares', 'diluted_shares']
}

for section_name, fields in sections.items():
    print(f"\n{section_name}:")
    print("-" * 80)
    section_df = income_df[income_df['Field'].isin(fields)]
    if not section_df.empty:
        print(section_df[['Status', 'Field', 'Value']].to_string(index=False))

---
## 5. Extract Balance Sheet
**Purpose:** Extract all balance sheet line items

**Expected:** 32-36 out of 38 fields for most companies

**Troubleshooting:** Same as income statement

In [ ]:
# Extract balance sheet
balance_df = await extract_balance_sheet(
    filing,
    TICKER,
    FILING_TYPE,
    YEAR,
    QUARTER,
    use_ai_fallback=USE_AI_FALLBACK
)

# Calculate coverage
balance_found = len(balance_df[balance_df['Status'] == '✓'])
balance_total = len(balance_df)
balance_coverage = (balance_found / balance_total) * 100

print(f"\n{'='*80}")
print(f"BALANCE SHEET RESULTS")
print(f"{'='*80}")
print(f"\nData Quality: {balance_found}/{balance_total} fields ({balance_coverage:.1f}%)")

# Show key metrics
print(f"\nKey Metrics:")
key_fields = ['total_current_assets', 'total_assets', 'total_current_liabilities', 
              'total_liabilities', 'total_equity']
for field in key_fields:
    row = balance_df[balance_df['Field'] == field]
    if not row.empty and pd.notna(row.iloc[0]['Value']):
        value = row.iloc[0]['Value']
        print(f"  {field:30s}: ${value:,.0f}")

In [ ]:
# Display balance sheet by section
print("\n" + "="*80)
print("BALANCE SHEET BREAKDOWN")
print("="*80)

sections = {
    "Current Assets": ['cash_and_equivalents', 'short_term_investments', 'accounts_receivable', 
                       'inventory', 'total_current_assets'],
    "Non-Current Assets": ['ppe_net', 'goodwill', 'intangible_assets', 'total_noncurrent_assets', 'total_assets'],
    "Current Liabilities": ['accounts_payable', 'short_term_debt', 'total_current_liabilities'],
    "Non-Current Liabilities": ['long_term_debt', 'total_noncurrent_liabilities', 'total_liabilities'],
    "Equity": ['common_stock', 'retained_earnings', 'total_stockholders_equity', 'total_equity']
}

for section_name, fields in sections.items():
    print(f"\n{section_name}:")
    print("-" * 80)
    section_df = balance_df[balance_df['Field'].isin(fields)]
    if not section_df.empty:
        print(section_df[['Status', 'Field', 'Value']].to_string(index=False))

---
## 6. Extract Cash Flow Statement
**Purpose:** Extract all cash flow line items

**Expected:** 25-28 out of 30 fields for most companies

**Troubleshooting:** Same as income statement

In [ ]:
# Extract cash flow statement
cashflow_df = await extract_cash_flow(
    filing,
    TICKER,
    FILING_TYPE,
    YEAR,
    QUARTER,
    use_ai_fallback=USE_AI_FALLBACK
)

# Calculate coverage
cashflow_found = len(cashflow_df[cashflow_df['Status'] == '✓'])
cashflow_total = len(cashflow_df)
cashflow_coverage = (cashflow_found / cashflow_total) * 100

print(f"\n{'='*80}")
print(f"CASH FLOW RESULTS")
print(f"{'='*80}")
print(f"\nData Quality: {cashflow_found}/{cashflow_total} fields ({cashflow_coverage:.1f}%)")

# Show key metrics
print(f"\nKey Metrics:")
key_fields = ['net_cash_operating_activities', 'net_cash_investing_activities', 
              'net_cash_financing_activities', 'net_change_in_cash', 'cash_end_of_period']
for field in key_fields:
    row = cashflow_df[cashflow_df['Field'] == field]
    if not row.empty and pd.notna(row.iloc[0]['Value']):
        value = row.iloc[0]['Value']
        print(f"  {field:35s}: ${value:,.0f}")

In [ ]:
# Display cash flow by section
print("\n" + "="*80)
print("CASH FLOW BREAKDOWN")
print("="*80)

sections = {
    "Operating Activities": ['net_income_starting_point', 'depreciation_amortization', 
                            'change_accounts_receivable', 'change_inventory', 'net_cash_operating_activities'],
    "Investing Activities": ['capital_expenditures', 'acquisitions', 'net_cash_investing_activities'],
    "Financing Activities": ['debt_issuance', 'debt_repayment', 'dividends_paid', 
                            'stock_repurchase', 'net_cash_financing_activities'],
    "Summary": ['net_change_in_cash', 'cash_beginning_of_period', 'cash_end_of_period']
}

for section_name, fields in sections.items():
    print(f"\n{section_name}:")
    print("-" * 80)
    section_df = cashflow_df[cashflow_df['Field'].isin(fields)]
    if not section_df.empty:
        print(section_df[['Status', 'Field', 'Value']].to_string(index=False))

---
## 7. Combined Analysis & Metrics
**Purpose:** Calculate cross-statement metrics and validation

**Examples:**
- Net income (income) should match net income starting point (cash flow)
- Cash end of period (cash flow) should match cash on balance sheet
- Total assets should equal liabilities + equity

In [ ]:
# Overall extraction summary
print("="*80)
print(f"EXTRACTION SUMMARY: {TICKER} {FILING_TYPE} {YEAR}")
print("="*80)

total_fields = income_total + balance_total + cashflow_total
total_found = income_found + balance_found + cashflow_found
overall_coverage = (total_found / total_fields) * 100

print(f"\nOverall Data Quality:")
print(f"  Total fields extracted: {total_found}/{total_fields} ({overall_coverage:.1f}%)")
print(f"\nBy Statement:")
print(f"  Income Statement:    {income_found}/{income_total} ({income_coverage:.1f}%)")
print(f"  Balance Sheet:       {balance_found}/{balance_total} ({balance_coverage:.1f}%)")
print(f"  Cash Flow:           {cashflow_found}/{cashflow_total} ({cashflow_coverage:.1f}%)")

# Quality assessment
print(f"\nQuality Assessment:")
if overall_coverage >= 85:
    print("  ✓ Excellent - High quality data")
elif overall_coverage >= 70:
    print("  ✓ Good - Acceptable quality")
elif overall_coverage >= 50:
    print("  ⚠ Fair - Some fields missing")
else:
    print("  ✗ Poor - Many fields missing")

In [ ]:
# Cross-statement validation
print("\n" + "="*80)
print("CROSS-STATEMENT VALIDATION")
print("="*80)

def get_value(df, field_name):
    """Helper to get value from dataframe"""
    row = df[df['Field'] == field_name]
    if not row.empty and pd.notna(row.iloc[0]['Value']):
        return row.iloc[0]['Value']
    return None

# Validation 1: Balance sheet equation
print("\n1. Balance Sheet Equation (Assets = Liabilities + Equity):")
total_assets = get_value(balance_df, 'total_assets')
total_liabilities = get_value(balance_df, 'total_liabilities')
total_equity = get_value(balance_df, 'total_equity')

if all([total_assets, total_liabilities, total_equity]):
    right_side = total_liabilities + total_equity
    variance = abs(total_assets - right_side) / total_assets * 100
    
    if variance < 0.01:
        print(f"  ✓ PASS - Variance: {variance:.4f}%")
    else:
        print(f"  ✗ FAIL - Variance: {variance:.4f}%")
    print(f"    Assets:              ${total_assets:,.0f}")
    print(f"    Liabilities + Equity: ${right_side:,.0f}")
else:
    print("  ⚠ SKIP - Missing values")

# Validation 2: Net income consistency
print("\n2. Net Income (Income Statement vs Cash Flow):")
net_income_is = get_value(income_df, 'net_income')
net_income_cf = get_value(cashflow_df, 'net_income_starting_point')

if all([net_income_is, net_income_cf]):
    variance = abs(net_income_is - net_income_cf) / net_income_is * 100
    
    if variance < 1.0:
        print(f"  ✓ PASS - Variance: {variance:.2f}%")
    else:
        print(f"  ⚠ WARNING - Variance: {variance:.2f}%")
    print(f"    Income Statement: ${net_income_is:,.0f}")
    print(f"    Cash Flow:        ${net_income_cf:,.0f}")
else:
    print("  ⚠ SKIP - Missing values")

# Validation 3: Cash reconciliation
print("\n3. Cash Balance (Balance Sheet vs Cash Flow):")
cash_bs = get_value(balance_df, 'cash_and_equivalents')
cash_cf = get_value(cashflow_df, 'cash_end_of_period')

if all([cash_bs, cash_cf]):
    variance = abs(cash_bs - cash_cf) / cash_bs * 100
    
    if variance < 1.0:
        print(f"  ✓ PASS - Variance: {variance:.2f}%")
    else:
        print(f"  ⚠ WARNING - Variance: {variance:.2f}%")
    print(f"    Balance Sheet: ${cash_bs:,.0f}")
    print(f"    Cash Flow:     ${cash_cf:,.0f}")
else:
    print("  ⚠ SKIP - Missing values")

In [ ]:
# Calculate key financial ratios
print("\n" + "="*80)
print("KEY FINANCIAL RATIOS")
print("="*80)

# Profitability Ratios
print("\nProfitability:")
revenue = get_value(income_df, 'revenue')
gross_profit = get_value(income_df, 'gross_profit')
operating_income = get_value(income_df, 'operating_income')
net_income = get_value(income_df, 'net_income')

if revenue and gross_profit:
    gross_margin = (gross_profit / revenue) * 100
    print(f"  Gross Margin:      {gross_margin:.1f}%")

if revenue and operating_income:
    operating_margin = (operating_income / revenue) * 100
    print(f"  Operating Margin:  {operating_margin:.1f}%")

if revenue and net_income:
    net_margin = (net_income / revenue) * 100
    print(f"  Net Margin:        {net_margin:.1f}%")

# Liquidity Ratios
print("\nLiquidity:")
current_assets = get_value(balance_df, 'total_current_assets')
current_liabilities = get_value(balance_df, 'total_current_liabilities')

if current_assets and current_liabilities:
    current_ratio = current_assets / current_liabilities
    print(f"  Current Ratio:     {current_ratio:.2f}")

# Leverage Ratios
print("\nLeverage:")
total_debt = get_value(balance_df, 'long_term_debt')

if total_debt and total_equity:
    debt_to_equity = total_debt / total_equity
    print(f"  Debt-to-Equity:    {debt_to_equity:.2f}")

if total_debt and total_assets:
    debt_ratio = total_debt / total_assets
    print(f"  Debt Ratio:        {debt_ratio:.2f}")

---
## 8. Export All Statements
**Purpose:** Save data to CSV and Excel files

**Output:**
- Individual CSV files for each statement
- Combined Excel workbook with all 3 statements
- Optional: Log to DuckDB

In [ ]:
# Export individual CSVs
print("="*80)
print("EXPORTING DATA")
print("="*80)

# Create filenames
fiscal_year = income_df.iloc[0]['Fiscal_Year']
filing_type_clean = income_df.iloc[0]['Filing_Type'].replace('/', '-')
quarter_str = f"_Q{income_df.iloc[0]['Quarter']}" if income_df.iloc[0]['Quarter'] else ""
base_filename = f"{TICKER}_{filing_type_clean}_{fiscal_year}{quarter_str}"

# Export CSVs
print("\nExporting CSV files...")
csv_files = {}

csv_files['income'] = f"{OUTPUT_DIR}/{base_filename}_income_statement.csv"
income_df.to_csv(csv_files['income'], index=False)
print(f"  ✓ {csv_files['income']}")

csv_files['balance'] = f"{OUTPUT_DIR}/{base_filename}_balance_sheet.csv"
balance_df.to_csv(csv_files['balance'], index=False)
print(f"  ✓ {csv_files['balance']}")

csv_files['cashflow'] = f"{OUTPUT_DIR}/{base_filename}_cash_flow.csv"
cashflow_df.to_csv(csv_files['cashflow'], index=False)
print(f"  ✓ {csv_files['cashflow']}")

In [ ]:
# Export combined Excel workbook
print("\nExporting Excel workbook...")
excel_file = f"{OUTPUT_DIR}/{base_filename}_all_statements.xlsx"

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    income_df.to_excel(writer, sheet_name='Income Statement', index=False)
    balance_df.to_excel(writer, sheet_name='Balance Sheet', index=False)
    cashflow_df.to_excel(writer, sheet_name='Cash Flow', index=False)
    
    # Create summary sheet
    summary_data = {
        'Metric': [
            'Ticker', 'Fiscal Year', 'Filing Type', 'Filing Date', 'Period End',
            '', 
            'Income Statement Coverage', 'Balance Sheet Coverage', 'Cash Flow Coverage',
            'Overall Coverage',
            '',
            'Revenue', 'Net Income', 'Total Assets', 'Total Equity', 'Operating Cash Flow'
        ],
        'Value': [
            TICKER, fiscal_year, filing_type_clean, 
            income_df.iloc[0]['Filing_Date'],
            income_df.iloc[0]['Period_End_Date'],
            '',
            f"{income_coverage:.1f}%", f"{balance_coverage:.1f}%", f"{cashflow_coverage:.1f}%",
            f"{overall_coverage:.1f}%",
            '',
            get_value(income_df, 'revenue'),
            get_value(income_df, 'net_income'),
            get_value(balance_df, 'total_assets'),
            get_value(balance_df, 'total_equity'),
            get_value(cashflow_df, 'net_cash_operating_activities')
        ]
    }
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_excel(writer, sheet_name='Summary', index=False)

print(f"  ✓ {excel_file}")
print(f"\n✓ All files exported successfully!")
print(f"\nFile sizes:")
for name, path in csv_files.items():
    size = os.path.getsize(path)
    print(f"  {name:10s}: {size:,} bytes")
excel_size = os.path.getsize(excel_file)
print(f"  {'excel':10s}: {excel_size:,} bytes")

In [ ]:
# Optional: Log to DuckDB
print("\n" + "="*80)
print("LOGGING TO DATABASE (Optional)")
print("="*80)

try:
    from xbrl_mapping_manager_multi_statement import XBRLMappingManager
    
    mapper = XBRLMappingManager(DB_PATH)
    
    filing_date_obj = date.fromisoformat(str(income_df.iloc[0]['Filing_Date']))
    
    # Log coverage for each statement
    await mapper.update_statement_coverage(
        ticker=TICKER,
        filing_date=filing_date_obj,
        filing_type=filing_type_clean,
        statement_type='income',
        fields_found=income_found,
        total_fields=income_total
    )
    print(f"✓ Logged income statement coverage to database")
    
    await mapper.update_statement_coverage(
        ticker=TICKER,
        filing_date=filing_date_obj,
        filing_type=filing_type_clean,
        statement_type='balance',
        fields_found=balance_found,
        total_fields=balance_total
    )
    print(f"✓ Logged balance sheet coverage to database")
    
    await mapper.update_statement_coverage(
        ticker=TICKER,
        filing_date=filing_date_obj,
        filing_type=filing_type_clean,
        statement_type='cashflow',
        fields_found=cashflow_found,
        total_fields=cashflow_total
    )
    print(f"✓ Logged cash flow coverage to database")
    
    # Get overall quality
    quality = await mapper.get_overall_quality(TICKER)
    print(f"\nOverall data quality in database: {quality['overall_coverage']:.1f}%")
    
    mapper.close()
    
except Exception as e:
    print(f"⚠ Database logging failed: {e}")
    print("  This is optional - files were still saved successfully")

---
## 9. Batch Processing (Multiple Tickers)
**Purpose:** Extract statements for multiple companies

**Usage:** Modify the TICKER_LIST below and run

**Note:** This can take 15-30 minutes for 10 companies

In [ ]:
# Batch processing configuration
TICKER_LIST = ['AAPL', 'MSFT', 'GOOGL', 'META', 'TSLA']  # Add more tickers here
BATCH_YEAR = 2024
BATCH_FILING_TYPE = '10-K'

print("="*80)
print("BATCH PROCESSING SETUP")
print("="*80)
print(f"\nTickers to process: {len(TICKER_LIST)}")
print(f"Filing type: {BATCH_FILING_TYPE}")
print(f"Year: {BATCH_YEAR}")
print(f"\nEstimated time: {len(TICKER_LIST) * 2} - {len(TICKER_LIST) * 3} minutes")
print(f"\nTickers: {', '.join(TICKER_LIST)}")

In [ ]:
# Run batch extraction
import time

batch_results = []
start_time = time.time()

print("\n" + "="*80)
print("BATCH EXTRACTION STARTING")
print("="*80)

for i, ticker in enumerate(TICKER_LIST, 1):
    print(f"\n[{i}/{len(TICKER_LIST)}] Processing {ticker}...")
    print("-" * 80)
    
    try:
        # Get filing
        filing = get_filing(ticker, BATCH_FILING_TYPE, BATCH_YEAR)
        
        # Extract all 3 statements
        income = await extract_income_statement(filing, ticker, BATCH_FILING_TYPE, BATCH_YEAR, use_ai_fallback=False)
        balance = await extract_balance_sheet(filing, ticker, BATCH_FILING_TYPE, BATCH_YEAR, use_ai_fallback=False)
        cashflow = await extract_cash_flow(filing, ticker, BATCH_FILING_TYPE, BATCH_YEAR, use_ai_fallback=False)
        
        # Calculate coverage
        income_cov = (len(income[income['Status'] == '✓']) / len(income)) * 100
        balance_cov = (len(balance[balance['Status'] == '✓']) / len(balance)) * 100
        cashflow_cov = (len(cashflow[cashflow['Status'] == '✓']) / len(cashflow)) * 100
        overall_cov = (income_cov + balance_cov + cashflow_cov) / 3
        
        # Save files
        base_name = f"{ticker}_{BATCH_FILING_TYPE.replace('/', '-')}_{BATCH_YEAR}"
        income.to_csv(f"{OUTPUT_DIR}/{base_name}_income.csv", index=False)
        balance.to_csv(f"{OUTPUT_DIR}/{base_name}_balance.csv", index=False)
        cashflow.to_csv(f"{OUTPUT_DIR}/{base_name}_cashflow.csv", index=False)
        
        # Store result
        batch_results.append({
            'ticker': ticker,
            'status': 'SUCCESS',
            'income_coverage': income_cov,
            'balance_coverage': balance_cov,
            'cashflow_coverage': cashflow_cov,
            'overall_coverage': overall_cov
        })
        
        print(f"  ✓ {ticker} complete - {overall_cov:.1f}% coverage")
        
    except Exception as e:
        print(f"  ✗ {ticker} failed: {e}")
        batch_results.append({
            'ticker': ticker,
            'status': 'FAILED',
            'error': str(e)
        })

elapsed = time.time() - start_time

print("\n" + "="*80)
print("BATCH EXTRACTION COMPLETE")
print("="*80)
print(f"\nTime elapsed: {elapsed/60:.1f} minutes")
print(f"Successful: {sum(1 for r in batch_results if r['status'] == 'SUCCESS')}/{len(TICKER_LIST)}")

# Show results
results_df = pd.DataFrame(batch_results)
print("\n" + results_df.to_string(index=False))

# Save batch summary
results_df.to_csv(f"{OUTPUT_DIR}/batch_summary_{BATCH_YEAR}.csv", index=False)
print(f"\n✓ Batch summary saved to: {OUTPUT_DIR}/batch_summary_{BATCH_YEAR}.csv")

---
## ✅ Extraction Complete!

### What was extracted:
- ✅ Income Statement
- ✅ Balance Sheet  
- ✅ Cash Flow Statement

### Where to find the data:
- CSV files: `./financial_statements/`
- Excel workbook: `./financial_statements/{TICKER}_{FILING_TYPE}_{YEAR}_all_statements.xlsx`
- Database: `data/xbrl_mappings_multi.duckdb`

### Next steps:
1. Review the data quality metrics above
2. Check for any failed validations
3. Analyze AI discoveries for potential mapping improvements
4. Use batch processing for multiple companies

---

In [ ]:
from financial_statements_db import FinancialStatementsDB

# Create/connect to database
db = FinancialStatementsDB('data/financial_statements.duckdb')

In [ ]:
from extractors.income_statement_extractor import get_filing, extract_income_statement
from extractors.balance_sheet_extractor import extract_balance_sheet
from extractors.cash_flow_extractor import extract_cash_flow

In [ ]:
filing = get_filing('AAPL', '10-K', 2025)
income_df = await extract_income_statement(filing, 'AAPL', '10-K')
balance_df = await extract_balance_sheet(filing, 'AAPL', '10-K')
cashflow_df = await extract_cash_flow(filing, 'AAPL', '10-K')

In [ ]:
db.insert_income_statement(income_df)
db.insert_balance_sheet(balance_df)
db.insert_cash_flow(cashflow_df)

In [ ]:
from financial_statements_db import FinancialStatementsDB

# Reload the module if already loaded
import importlib
import sys
if 'financial_statements_db' in sys.modules:
    del sys.modules['financial_statements_db']


In [ ]:

from financial_statements_db import FinancialStatementsDB

# Now insert
db = FinancialStatementsDB('data/financial_statements.duckdb')
results = db.insert_all_statements(income_df, balance_df, cashflow_df)


In [ ]:
statements = db.get_company_statements('AAPL')
print(statements['income'])
print(statements['balance'])
print(statements['cashflow'])

In [ ]:
income_only = db.get_company_statements('AAPL', statement_type='income')

In [ ]:
revenue_ts = db.get_time_series('AAPL', 'income', 'revenue')
print(revenue_ts)

In [4]:
import asyncio
from bulk_import_10k import bulk_import_10k

# Run bulk import
results = await bulk_import_10k(
    ticker_csv='data/import_tickers.csv',
    periods=5,                              # Last 5 years
    db_path='data/financial_statements.duckdb',
    use_ai_fallback=True,                   # Set True for better coverage
    skip_existing=True,                      # Skip already-imported filings
    rate_limit_delay=1.0                     # 1 second between requests
)

python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 5


✓ Loaded comprehensive XBRL mapping (125 concepts, 30 fields)
✓ Loaded balance sheet XBRL mapping (62 concepts, 37 fields)
✓ Loaded cash flow XBRL mapping (50 concepts, 31 fields)
BULK 10-K IMPORT

📋 Reading tickers from data/import_tickers.csv...
✓ Found 124 tickers

💾 Connecting to database: data/financial_statements.duckdb...
✓ Database schema created/verified
✓ Connected to financial statements database: data/financial_statements.duckdb

🚀 Starting bulk extraction...
   Rate limit: 1.0s between requests
   AI fallback: Enabled
   Skip existing: Yes


[1/124] Processing AAPL...
--------------------------------------------------------------------------------

✓ AAPL complete:
  Filings found: 5
  Filings processed: 0
  Filings skipped: 5
  Filings failed: 0
  Statements success: 0
  Statements failed: 0
  Duration: 1.0s

[2/124] Processing ABT...
--------------------------------------------------------------------------------

EXTRACTING INCOME STATEMENT

✓ Available periods: 2024-12

CancelledError: 

In [ ]:
import asyncio
from xbrl_concept_mapper_ollama import get_statement_mapping, check_ollama_available

# Check if Ollama is ready
ready = await check_ollama_available()
if not ready:
    print("Ollama not ready - check setup!")

# Test mapping
result = await get_statement_mapping("SellingAndMarketingExpense", "income")
print(result)  # Should print: "selling_general_admin"import asyncio
from xbrl_concept_mapper_ollama import get_statement_mapping, check_ollama_available

# Check if Ollama is ready
ready = await check_ollama_available()
if not ready:
    print("Ollama not ready - check setup!")

# Test mapping
result = await get_statement_mapping("SellingAndMarketingExpense", "income")
print(result)  # Should print: "selling_general_admin"

In [3]:
import asyncio
from xbrl_concept_mapper_ollama import get_statement_mapping, check_ollama_available

# Check if Ollama is ready
ready = await check_ollama_available()
if not ready:
    print("Ollama not ready - check setup!")
else:
    # Test mapping
    result = await get_statement_mapping("SellingAndMarketingExpense", "income")
    print(f"Result: {result}")  # Should print: "selling_general_admin"


Checking Ollama availability...
  URL: http://172.17.112.1:11434
  Model: deepseek-r1:8b
✅ Ollama ready with model: deepseek-r1:8b
Mapper called: SellingAndMarketingExpense (income) → already_mapped
Result: already_mapped


In [ ]:
import asyncio
from xbrl_concept_mapper_ollama import get_statement_mapping, check_ollama_available

# Check Ollama
ready = await check_ollama_available()

if ready:
    print("\n" + "="*80)
    print("TESTING OLLAMA XBRL MAPPER")
    print("="*80)
    
    # Test multiple concepts
    test_cases = [
        ("SellingAndMarketingExpense", "income", "selling_general_admin"),
        ("ResearchAndDevelopmentExpense", "income", "research_development"),
        ("PropertyPlantAndEquipmentNet", "balance", "ppe_net"),
        ("PaymentsToAcquirePropertyPlantAndEquipment", "cashflow", "capital_expenditures"),
    ]
    
    for concept, stmt_type, expected in test_cases:
        print(f"\nTesting: {concept}")
        result = await get_statement_mapping(concept, stmt_type)
        status = "✅" if result == expected else "❌"
        print(f"{status} Expected: {expected}, Got: {result}")
    
    print("\n" + "="*80)
    print("Tests complete!")
    print("="*80)
else:
    print("❌ Ollama not ready!")
    print("\nTo fix:")
    print("1. Install: curl -fsSL https://ollama.ai/install.sh | sh")
    print("2. Pull model: ollama pull deepseek-r1:8b")
    print("3. Start: ollama serve")

In [ ]:
import asyncio
import time
from xbrl_concept_mapper_ollama import get_statement_mapping, check_ollama_available

ready = await check_ollama_available()

if ready:
    print("Benchmarking Ollama mapper speed...")
    
    # Test 10 mappings
    concepts = [
        "Revenue", "CostOfRevenue", "GrossProfit", 
        "OperatingExpenses", "InterestExpense", "IncomeTaxExpense",
        "NetIncome", "EarningsPerShare", "Assets", "Liabilities"
    ]
    
    start = time.time()
    
    for concept in concepts:
        result = await get_statement_mapping(concept, "income")
        print(f"  {concept} → {result}")
    
    elapsed = time.time() - start
    avg_time = elapsed / len(concepts)
    
    print(f"\nTotal time: {elapsed:.1f}s")
    print(f"Average per call: {avg_time:.2f}s")
    print(f"Estimated bulk import (2000 calls): {(avg_time * 2000)/60:.0f} minutes")


In [2]:
import asyncio
from xbrl_concept_mapper_ollama import check_ollama_available, get_statement_mapping

async def test():
    # Test connection
    print("Testing Ollama connection...\n")
    ready = await check_ollama_available()
    
    if ready:
        print("\n" + "="*80)
        print("Testing mapping...")
        print("="*80)
        
        # Test a few mappings
        test_cases = [
            ("Revenue", "income"),
            ("CostOfRevenue", "income"),
            ("PropertyPlantAndEquipmentNet", "balance"),
        ]
        
        for concept, stmt_type in test_cases:
            result = await get_statement_mapping(concept, stmt_type)
            print(f"  {concept} ({stmt_type}) → {result}")
        
        print("\n✅ Ollama mapper is working!")
        return True
    else:
        print("\n❌ Ollama not available")
        return False

# Run test
await test()


python-dotenv could not parse statement starting at line 5


Detected WSL Windows host IP: 172.17.112.1
Ollama configuration: http://172.17.112.1:11434 (model: deepseek-r1:8b)
Testing Ollama connection...


Checking Ollama availability...
  URL: http://172.17.112.1:11434
  Model: deepseek-r1:8b
✅ Ollama ready with model: deepseek-r1:8b

Testing mapping...
Mapper called: Revenue (income) → already_mapped
  Revenue (income) → already_mapped
Mapper called: CostOfRevenue (income) → already_mapped
  CostOfRevenue (income) → already_mapped
Mapper called: PropertyPlantAndEquipmentNet (balance) → already_mapped
  PropertyPlantAndEquipmentNet (balance) → already_mapped

✅ Ollama mapper is working!


True

In [ ]:
import os
import sys

print("Current working directory:", os.getcwd())
print("\nPython path:")
for p in sys.path:
    print(f"  {p}")

print("\nChecking xbrl_mappings folder:")
xbrl_path = os.path.join(os.getcwd(), "xbrl_mappings")
print(f"  Path: {xbrl_path}")
print(f"  Exists: {os.path.exists(xbrl_path)}")

if os.path.exists(xbrl_path):
    print(f"\n  Contents:")
    for f in os.listdir(xbrl_path):
        print(f"    - {f}")
        
print("\nTrying to import:")
try:
    from xbrl_concept_mapper_ollama import read_mapping_file
    print("  ✅ Import successful")
except Exception as e:
    print(f"  ❌ Import failed: {e}")